# diffusion analysis: emeralds and tomatoes

this notebook performs a thorough statistical analysis of two assets -- emeralds and tomatoes --
using order book and trade data from two trading days (day -2 and day -1).

the goal is to:
- characterise the statistical nature of each price series (mean-reverting vs. trending vs. random walk)
- estimate the rate at which deviations from equilibrium decay (half-life of mean reversion)
- measure how much randomness vs. directionality drives price changes (diffusion entropy, drift-to-diffusion ratio)
- detect structural breaks or regime changes over time
- derive actionable trading signals: when to enter, when to exit, and how confident we should be

the two assets behave very differently. this will become obvious quickly, and every analysis
is designed to make that contrast interpretable and monetisable.

note on terminology throughout this notebook:
- "diffusion" refers to the random, noise-driven component of price movement (the sigma in a stochastic model)
- "drift" refers to the systematic, directional component (the mu)
- "mean reversion" means the price is pulled back toward some central value after deviating from it
- "half-life" is the expected time for a deviation to decay to half its original size


## section 1: imports and configuration

we use only the standard scientific python stack (numpy, scipy, pandas, matplotlib).
all statistical routines are implemented from scratch so the methodology is fully transparent.


In [3]:
import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.optimize import minimize_scalar, minimize
import warnings
warnings.filterwarnings('ignore')

# reproducibility
np.random.seed(42)

# plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f8f8',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

ASSETS = ['EMERALDS', 'TOMATOES']
COLORS = {'EMERALDS': '#2196F3', 'TOMATOES': '#E91E63'}


## section 2: data loading

prices are sampled at regular 100-unit timestamp intervals. trades are event-driven
(they only appear when a transaction occurs).

we combine both days into a single continuous series per asset.
the timestamp resets each day, so we create a global time index.

the key column we work with is `mid_price`, defined as the average of the best
bid and best ask. this is the most neutral estimate of fair value at each moment.


In [4]:
# --- load price data ---
p1 = pd.read_csv('prices_round_0_day_-1.csv', sep=';')
p2 = pd.read_csv('prices_round_0_day_-2.csv', sep=';')
prices_raw = pd.concat([p2, p1], ignore_index=True)

# sort by day then timestamp within day
prices_raw = prices_raw.sort_values(['day', 'timestamp']).reset_index(drop=True)

# --- load trade data ---
t1 = pd.read_csv('trades_round_0_day_-1.csv', sep=';')
t2 = pd.read_csv('trades_round_0_day_-2.csv', sep=';')
trades_raw = pd.concat([t2, t1], ignore_index=True)
trades_raw['day'] = trades_raw.get('day', np.where(trades_raw.index < len(t2), -2, -1))
# assign day column to trades based on which file they came from
t2['day'] = -2
t1['day'] = -1
trades_raw = pd.concat([t2, t1], ignore_index=True)
trades_raw = trades_raw.sort_values(['day', 'timestamp']).reset_index(drop=True)

# --- build per-asset mid-price series ---
# each day has 10000 timestamps (0, 100, 200, ..., 999900)
# we create a global tick index: day -2 contributes ticks 0..9999, day -1 contributes 10000..19999

asset_data = {}
for asset in ASSETS:
    sub = prices_raw[prices_raw['product'] == asset].copy()
    sub = sub.sort_values(['day', 'timestamp']).reset_index(drop=True)
    # global tick index
    day_offset = {-2: 0, -1: 10000}
    sub['tick'] = sub.apply(lambda r: day_offset[r['day']] + r['timestamp'] // 100, axis=1)
    sub = sub.set_index('tick')
    asset_data[asset] = sub

    tr = trades_raw[trades_raw['symbol'] == asset].copy()
    tr['day_offset'] = tr['day'].map(day_offset)
    tr['tick'] = tr['day_offset'] + tr['timestamp'] // 100
    asset_data[f'{asset}_trades'] = tr

print("price data shape per asset:")
for a in ASSETS:
    df = asset_data[a]
    print(f"  {a}: {len(df)} rows, mid_price range [{df['mid_price'].min():.1f}, {df['mid_price'].max():.1f}]")

print()
print("trade data:")
for a in ASSETS:
    tr = asset_data[f'{a}_trades']
    print(f"  {a}: {len(tr)} trades, price range [{tr['price'].min():.1f}, {tr['price'].max():.1f}]")


FileNotFoundError: [Errno 2] No such file or directory: 'prices_round_0_day_-1.csv'

## section 3: exploratory data analysis

before any modelling, we inspect the raw price series visually and quantitatively.
this section answers: what does each asset look like? how volatile is it? how wide is the spread?

the bid-ask spread is important because it represents the minimum round-trip cost of trading.
any mean-reversion signal must generate moves larger than the spread to be profitable.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for i, asset in enumerate(ASSETS):
    df = asset_data[asset]
    mp = df['mid_price']
    spread = df['ask_price_1'] - df['bid_price_1']
    tr = asset_data[f'{asset}_trades']

    # mid-price over time
    ax = axes[i, 0]
    ax.plot(mp.index, mp.values, color=COLORS[asset], linewidth=0.6, alpha=0.9, label='mid price')
    ax.axvline(10000, color='gray', linestyle='--', alpha=0.5, linewidth=1, label='day boundary')
    if len(tr) > 0:
        ax.scatter(tr['tick'], tr['price'], color='black', s=6, zorder=5, alpha=0.5, label='trades')
    ax.set_title(f'{asset.lower()} -- mid price over time')
    ax.set_xlabel('global tick (100-unit steps)')
    ax.set_ylabel('price')
    ax.legend(fontsize=9)

    # spread distribution
    ax = axes[i, 1]
    ax.hist(spread.dropna(), bins=40, color=COLORS[asset], alpha=0.7, edgecolor='white')
    ax.axvline(spread.mean(), color='black', linestyle='--', linewidth=1.5, label=f'mean = {spread.mean():.2f}')
    ax.set_title(f'{asset.lower()} -- bid-ask spread distribution (ask1 - bid1)')
    ax.set_xlabel('spread (price units)')
    ax.set_ylabel('frequency')
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# summary statistics
print("=" * 60)
print("summary statistics -- mid price")
print("=" * 60)
for asset in ASSETS:
    df = asset_data[asset]
    mp = df['mid_price']
    spread = df['ask_price_1'] - df['bid_price_1']
    diffs = mp.diff().dropna()
    print(f"\n{asset.lower()}")
    print(f"  observations      : {len(mp):,}")
    print(f"  mean price        : {mp.mean():.4f}")
    print(f"  std of price      : {mp.std():.4f}")
    print(f"  min / max         : {mp.min():.1f} / {mp.max():.1f}")
    print(f"  mean spread       : {spread.mean():.4f}")
    print(f"  std of spread     : {spread.std():.4f}")
    print(f"  mean abs increment: {diffs.abs().mean():.4f}")
    print(f"  std of increments : {diffs.std():.4f}")
    print(f"  skew of increments: {diffs.skew():.4f}")
    print(f"  kurt of increments: {diffs.kurtosis():.4f}  (excess kurtosis; normal = 0)")


## section 4: increment analysis and normality

price increments (changes between consecutive ticks) are the raw material of diffusion models.
a gaussian distribution of increments is the baseline assumption for a brownian-motion process.

we inspect:
- the distribution shape (histogram + qq-plot)
- kurtosis (heavy tails mean large jumps happen more often than gaussian would predict)
- autocorrelation of increments (non-zero autocorrelation breaks the random-walk assumption)

if increments are strongly autocorrelated and negative at lag 1, it implies mean reversion
at the timescale of one tick. positive lag-1 autocorrelation implies momentum.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, asset in enumerate(ASSETS):
    mp = asset_data[asset]['mid_price']
    diffs = mp.diff().dropna().values

    # histogram
    ax = axes[i, 0]
    x_range = np.linspace(diffs.min(), diffs.max(), 200)
    mu_d, sigma_d = diffs.mean(), diffs.std()
    ax.hist(diffs, bins=60, density=True, color=COLORS[asset], alpha=0.6, edgecolor='white')
    ax.plot(x_range, stats.norm.pdf(x_range, mu_d, sigma_d), 'k-', linewidth=2, label='fitted normal')
    ax.set_title(f'{asset.lower()} -- increment distribution')
    ax.set_xlabel('price increment')
    ax.set_ylabel('density')
    ax.legend()

    # qq plot
    ax = axes[i, 1]
    (q_sample, q_theory), _ = stats.probplot(diffs, dist='norm')
    ax.scatter(q_theory, q_sample, color=COLORS[asset], s=3, alpha=0.4)
    lims = [min(q_theory), max(q_theory)]
    ax.plot(lims, lims, 'k--', linewidth=1.5, label='perfect normal')
    ax.set_title(f'{asset.lower()} -- qq plot of increments')
    ax.set_xlabel('theoretical quantiles')
    ax.set_ylabel('sample quantiles')
    ax.legend()

    # autocorrelation of increments
    ax = axes[i, 2]
    max_lag = 40
    acf_vals = [pd.Series(diffs).autocorr(lag=k) for k in range(1, max_lag+1)]
    lags = np.arange(1, max_lag+1)
    conf = 1.96 / np.sqrt(len(diffs))
    ax.bar(lags, acf_vals, color=COLORS[asset], alpha=0.7, width=0.7)
    ax.axhline(conf, color='red', linestyle='--', linewidth=1, label=f'95% ci (+/-{conf:.3f})')
    ax.axhline(-conf, color='red', linestyle='--', linewidth=1)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_title(f'{asset.lower()} -- autocorrelation of increments')
    ax.set_xlabel('lag')
    ax.set_ylabel('autocorrelation')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# shapiro-wilk normality test on a subsample (full series is too large for shapiro)
print("normality tests (shapiro-wilk on 500-sample subsample):")
for asset in ASSETS:
    mp = asset_data[asset]['mid_price']
    diffs = mp.diff().dropna().values
    sample = np.random.choice(diffs, 500, replace=False)
    stat, pval = stats.shapiro(sample)
    print(f"  {asset.lower()}: W={stat:.4f}, p={pval:.4f}  -> {'reject normality' if pval < 0.05 else 'cannot reject normality'}")

print()
print("lag-1 autocorrelation of increments:")
for asset in ASSETS:
    mp = asset_data[asset]['mid_price']
    diffs = mp.diff().dropna()
    acf1 = diffs.autocorr(lag=1)
    n = len(diffs)
    se = 1.0 / np.sqrt(n)
    z = acf1 / se
    pval = 2 * (1 - stats.norm.cdf(abs(z)))
    print(f"  {asset.lower()}: acf(1) = {acf1:.5f}, z = {z:.2f}, p = {pval:.4f}")


## section 5: diffusion parameter estimation

we model each price series as a continuous-time process with two components:
- drift (mu): a constant directional pull per unit time
- diffusion (sigma): the standard deviation of random fluctuations per unit time

in discrete form at sampling interval dt:

    p(t+dt) = p(t) + mu * dt + sigma * sqrt(dt) * Z,  where Z ~ Normal(0,1)

this is the simplest possible model (arithmetic brownian motion). it gives us a baseline.

maximum likelihood estimation (mle) is used to estimate mu and sigma from observed increments.

the key insight is:
- if sigma >> |mu|, the process is essentially random (noise dominates)
- if |mu| / sigma is large, there is a meaningful directional trend

limitations of this model:
- it assumes constant parameters over the entire period (no regime changes)
- it assumes independence of increments (no autocorrelation)
- it is purely descriptive at this stage, not causal


In [ ]:
def estimate_diffusion_params(series, dt=1.0):
    """
    estimate drift (mu) and diffusion (sigma) from a price series
    using maximum likelihood for arithmetic brownian motion.

    the mle estimators are simply:
        mu_hat    = mean(increments) / dt
        sigma_hat = std(increments)  / sqrt(dt)

    returns: mu, sigma, and their standard errors
    """
    increments = np.diff(np.array(series, dtype=float))
    n = len(increments)
    mu_hat = increments.mean() / dt
    sigma_hat = increments.std(ddof=1) / np.sqrt(dt)
    # standard errors via delta method
    se_mu = sigma_hat / np.sqrt(n * dt)
    se_sigma = sigma_hat / np.sqrt(2 * (n - 1))
    return {
        'mu': mu_hat,
        'sigma': sigma_hat,
        'se_mu': se_mu,
        'se_sigma': se_sigma,
        'n_increments': n,
        'dt': dt
    }

print("diffusion parameter estimates (arithmetic brownian motion, dt = 1 tick):")
print("=" * 70)
diffusion_params = {}
for asset in ASSETS:
    mp = asset_data[asset]['mid_price'].values
    params = estimate_diffusion_params(mp)
    diffusion_params[asset] = params
    print(f"\n{asset.lower()}")
    print(f"  drift  mu    : {params['mu']:+.6f}  (se = {params['se_mu']:.6f})")
    print(f"  diffusion sigma: {params['sigma']:.6f}  (se = {params['se_sigma']:.6f})")
    print(f"  |mu| / sigma   : {abs(params['mu']) / params['sigma']:.6f}  (drift-to-diffusion ratio per tick)")
    t_stat = params['mu'] / params['se_mu']
    pval = 2 * (1 - stats.norm.cdf(abs(t_stat)))
    print(f"  drift t-stat   : {t_stat:.4f}  (p = {pval:.4f})  -> drift is {'significant' if pval < 0.05 else 'not significant'}")


## section 6: hurst exponent -- fractal scaling

the hurst exponent H characterises the long-run memory structure of a time series:

    H = 0.5  -> random walk (no memory, pure brownian motion)
    H < 0.5  -> mean reverting (anti-persistent, past gains tend to be followed by losses)
    H > 0.5  -> trending (persistent, past gains tend to continue)

we estimate H using rescaled range (r/s) analysis:
1. divide the series into non-overlapping windows of size n
2. for each window, compute the range of cumulative deviations from the mean
3. normalise by the standard deviation within that window (this is the "rescaled range")
4. regress log(r/s) on log(n): the slope is H

interpretation:
- emeralds with H close to 0 or very low would confirm extremely strong mean reversion
- tomatoes with H near 0.5 would suggest a nearly random walk over the observed period

limitations:
- r/s analysis can be biased for short-memory processes and with small sample sizes
- our series length of 20,000 gives reasonable but not definitive estimates


important warning about r/s analysis on these specific series:

both assets have strongly negative lag-1 autocorrelation in increments (around -0.49 for emeralds,
-0.42 for tomatoes). this is a known failure mode for r/s analysis. the rescaled range statistic
assumes increments are approximately iid; when they have strong autocorrelation, the estimated H
is biased. in particular, strong negative autocorrelation at short lags can produce H > 1 for
tomatoes even though H is mathematically bounded between 0 and 1.

for series with autocorrelated increments, the detrended fluctuation analysis (dfa) method is
more robust than r/s. we do not have dfa available here, so treat hurst estimates as indicative
only. the adf test and the ou model fitting are more reliable diagnostics for these series.


In [ ]:
def hurst_rs(series, min_window=10, max_window=None, n_points=20):
    """
    estimate the hurst exponent using rescaled range (r/s) analysis.
    returns H, the log-log regression results, and diagnostic arrays.
    """
    series = np.array(series, dtype=float)
    n = len(series)
    if max_window is None:
        max_window = n // 4

    window_sizes = np.unique(np.logspace(np.log10(min_window), np.log10(max_window), n_points).astype(int))
    rs_means = []

    for w in window_sizes:
        n_chunks = n // w
        if n_chunks < 2:
            continue
        rs_vals = []
        for k in range(n_chunks):
            chunk = series[k*w:(k+1)*w]
            mean_c = chunk.mean()
            std_c = chunk.std(ddof=1)
            if std_c == 0:
                continue
            dev = np.cumsum(chunk - mean_c)
            r = dev.max() - dev.min()
            rs_vals.append(r / std_c)
        if len(rs_vals) > 0:
            rs_means.append((w, np.mean(rs_vals)))

    ws = np.array([x[0] for x in rs_means], dtype=float)
    rs = np.array([x[1] for x in rs_means], dtype=float)
    log_w = np.log(ws)
    log_rs = np.log(rs)
    H, intercept, r_val, _, se_H = stats.linregress(log_w, log_rs)
    return H, se_H, r_val**2, ws, rs

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hurst_results = {}
for i, asset in enumerate(ASSETS):
    mp = asset_data[asset]['mid_price'].values
    H, se_H, r2, ws, rs = hurst_rs(mp)
    hurst_results[asset] = {'H': H, 'se': se_H, 'r2': r2}

    ax = axes[i]
    ax.scatter(np.log(ws), np.log(rs), color=COLORS[asset], s=40, zorder=5, label='observed r/s')
    fit_line = H * np.log(ws) + (np.log(rs).mean() - H * np.log(ws).mean())
    ax.plot(np.log(ws), fit_line, 'k--', linewidth=2, label=f'H = {H:.4f} (se={se_H:.4f})')
    ax.axline((0, 0), slope=0.5, color='gray', linestyle=':', linewidth=1.5, label='H=0.5 (random walk)')
    ax.set_title(f'{asset.lower()} -- hurst exponent via r/s analysis')
    ax.set_xlabel('log(window size)')
    ax.set_ylabel('log(rescaled range)')
    ax.legend()

plt.tight_layout()
plt.show()

print("hurst exponent results:")
for asset in ASSETS:
    r = hurst_results[asset]
    if r['H'] > 1.0:
        interp = "invalid -- H > 1 is impossible; r/s method unreliable here (see section 17)"
    elif r['H'] < 0.45:
        interp = "mean reverting (anti-persistent)"
    elif r['H'] > 0.55:
        interp = "trending (persistent) -- but verify with other tests"
    else:
        interp = "near random walk"
    print(f"  {asset.lower()}: H = {r['H']:.4f} +/- {r['se']:.4f}, R2 = {r['r2']:.4f}  -> {interp}")


## section 7: augmented dickey-fuller (adf) test -- stationarity

the adf test asks: is there a unit root in the price series?

a unit root means the series is non-stationary (integrated of order 1, i.e., a random walk or trend).
stationarity means the series fluctuates around a fixed mean -- which is the necessary condition
for mean-reversion trading strategies to work.

the test regresses the first difference of the series on the lagged level and lagged differences:

    delta_p(t) = alpha + beta * p(t-1) + sum_j gamma_j * delta_p(t-j) + epsilon(t)

null hypothesis: beta = 0 (unit root, non-stationary)
alternative: beta < 0 (mean reverting, stationary)

we use approximate critical values from the fuller (1976) tables:
- 1% critical value: -3.43
- 5% critical value: -2.86
- 10% critical value: -2.57

a t-statistic more negative than these thresholds rejects the null.

limitations:
- the adf test has low power in small samples and with near-unit-root processes
- the choice of lag order affects the result; we use lag=5 as a moderate choice
- we do not use exact p-values here (requires interpolation of fuller's tables); we compare to critical values


In [ ]:
def manual_adf(series, lags=5, include_constant=True):
    """
    augmented dickey-fuller test implemented from scratch.
    
    returns:
        t_stat  : test statistic for the lagged level coefficient
        beta    : estimated ar coefficient on the lagged level
        resid   : regression residuals
    
    the more negative the t_stat, the stronger the evidence against a unit root.
    critical values (approximate, from fuller 1976, with constant, n~infinity):
        1%  : -3.43
        5%  : -2.86
        10% : -2.57
    """
    y = np.array(series, dtype=float)
    dy = np.diff(y)           # length n-1
    n = len(dy)

    dy_t  = dy[lags:]         # target variable
    y_lag1 = y[lags:-1]       # y_{t-1}

    cols = [y_lag1]
    if include_constant:
        cols.append(np.ones(len(dy_t)))
    for j in range(1, lags + 1):
        cols.append(dy[lags - j : n - j])

    X = np.column_stack(cols)
    beta, _, _, _ = np.linalg.lstsq(X, dy_t, rcond=None)
    resid = dy_t - X @ beta
    dof = len(dy_t) - X.shape[1]
    s2 = np.dot(resid, resid) / dof
    cov = s2 * np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(cov))
    t_stat = beta[0] / se[0]
    return t_stat, beta[0], resid

# adf critical values (constant included, large n)
ADF_CV = {0.01: -3.43, 0.05: -2.86, 0.10: -2.57}

print("augmented dickey-fuller test (lags = 5, with constant)")
print("null hypothesis: unit root (non-stationary)")
print("=" * 65)
adf_results = {}
for asset in ASSETS:
    mp = asset_data[asset]['mid_price'].values
    t_stat, beta, resid = manual_adf(mp, lags=5)
    adf_results[asset] = {'t': t_stat, 'beta': beta}

    print(f"\n{asset.lower()}")
    print(f"  t-statistic : {t_stat:.4f}")
    print(f"  beta (ar)   : {beta:.6f}")
    for sig, cv in ADF_CV.items():
        flag = "reject unit root" if t_stat < cv else "fail to reject"
        print(f"  at {int(sig*100)}% level (cv={cv}): {flag}")


## section 8: ornstein-uhlenbeck model and half-life of mean reversion

the ornstein-uhlenbeck (ou) process is the canonical continuous-time mean-reversion model:

    dp(t) = theta * (mu - p(t)) * dt + sigma * dW(t)

where:
- mu    is the long-run equilibrium price (the level the process is pulled toward)
- theta is the speed of mean reversion (how fast deviations decay)
- sigma is the volatility of random shocks
- dW(t) is a brownian motion increment

the half-life is derived from theta:

    half_life = ln(2) / theta

this tells you: if the price is x units away from mu, after half_life ticks
it will on average be x/2 units away. this is the most practical summary
of mean-reversion speed.

we estimate theta by regressing p(t) - p(t-1) on p(t-1) - mu_hat:

    delta_p(t) = -theta * (p(t-1) - mu) + epsilon(t)

where mu is estimated as the sample mean.

trading interpretation:
- short half-life (say < 50 ticks) means mean reversion is fast enough to trade profitably
  at the 100-unit sampling frequency we have
- long half-life (say > 1000 ticks) means mean reversion exists but is slow and risky to trade

limitations:
- assumes the equilibrium mu is constant; if mu shifts, the estimate is biased
- theta and sigma are assumed constant (no regime changes)
- the discrete-time regression approximates the continuous-time ou; this is valid for small dt


In [ ]:
def fit_ou_process(series):
    """
    fit an ornstein-uhlenbeck process by ordinary least squares.

    the discrete approximation is:
        p(t) - p(t-1) = a + b * p(t-1) + epsilon

    where:
        b  = -theta * dt  (so theta = -b / dt, with dt = 1 tick)
        a  = theta * mu * dt  (so mu = -a / b)

    returns a dict with theta, mu, sigma, half_life, and diagnostics.
    """
    p = np.array(series, dtype=float)
    dp = np.diff(p)               # delta p
    p_lag = p[:-1]                # p(t-1)

    # ols: dp = a + b * p_lag
    X = np.column_stack([np.ones(len(dp)), p_lag])
    beta, _, _, _ = np.linalg.lstsq(X, dp, rcond=None)
    a, b = beta

    resid = dp - X @ beta
    n = len(dp)
    dof = n - 2
    s2 = np.dot(resid, resid) / dof
    cov = s2 * np.linalg.inv(X.T @ X)
    se = np.sqrt(np.diag(cov))

    theta = -b                       # mean-reversion speed (per tick)
    mu_eq = -a / b if b != 0 else np.nan  # equilibrium level
    sigma_ou = np.sqrt(s2)           # residual volatility

    if theta > 0:
        half_life = np.log(2) / theta
    else:
        half_life = np.inf           # non-mean-reverting

    # t-stat for b (is mean reversion significantly different from zero?)
    t_b = b / se[1]

    return {
        'theta': theta,
        'mu_eq': mu_eq,
        'sigma_ou': sigma_ou,
        'half_life': half_life,
        'b': b,
        'a': a,
        'se_b': se[1],
        't_b': t_b,
        'residuals': resid,
    }

ou_results = {}
for asset in ASSETS:
    mp = asset_data[asset]['mid_price'].values
    res = fit_ou_process(mp)
    ou_results[asset] = res

print("ornstein-uhlenbeck model results")
print("=" * 60)
for asset in ASSETS:
    r = ou_results[asset]
    print(f"\n{asset.lower()}")
    print(f"  equilibrium mu    : {r['mu_eq']:.4f}")
    print(f"  reversion speed theta : {r['theta']:.6f} per tick")
    print(f"  volatility sigma  : {r['sigma_ou']:.4f}")
    print(f"  half-life         : {r['half_life']:.2f} ticks")
    print(f"  t-stat on b       : {r['t_b']:.4f}  (b = {r['b']:.6f}, se = {r['se_b']:.6f})")
    pval_b = 2 * (1 - stats.norm.cdf(abs(r['t_b'])))
    print(f"  p-value on b      : {pval_b:.6f}  -> mean reversion {'is' if pval_b < 0.05 else 'is not'} statistically significant")
    if np.isfinite(r['half_life']):
        if r['half_life'] < 100:
            print(f"  interpretation    : very fast mean reversion -- strong intraday signal potential")
        elif r['half_life'] < 1000:
            print(f"  interpretation    : moderate mean reversion -- multi-day positions needed")
        else:
            print(f"  interpretation    : slow mean reversion -- hard to exploit at tick frequency")
    else:
        print(f"  interpretation    : no mean reversion detected (theta <= 0)")


In [ ]:
# visualise the mean-reversion fit
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for i, asset in enumerate(ASSETS):
    mp = asset_data[asset]['mid_price'].values
    r = ou_results[asset]
    dp = np.diff(mp)
    p_lag = mp[:-1]

    # scatter: p(t-1) vs delta_p with ou fit line
    ax = axes[i, 0]
    ax.scatter(p_lag, dp, color=COLORS[asset], s=2, alpha=0.15)
    xlim = np.array([p_lag.min(), p_lag.max()])
    ax.plot(xlim, r['a'] + r['b'] * xlim, 'k-', linewidth=2, label=f'ou fit (theta={r["theta"]:.5f})')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(r['mu_eq'], color='red', linestyle='--', linewidth=1.5, label=f'equilibrium = {r["mu_eq"]:.2f}')
    ax.set_title(f'{asset.lower()} -- ou scatter: p(t-1) vs delta_p')
    ax.set_xlabel('p(t-1)')
    ax.set_ylabel('delta_p(t)')
    ax.legend(fontsize=9)

    # simulate ou path from ou params and compare to actual
    ax = axes[i, 1]
    n = len(mp)
    sim = np.zeros(n)
    sim[0] = mp[0]
    for t in range(1, n):
        sim[t] = sim[t-1] + r['a'] + r['b'] * sim[t-1] + r['sigma_ou'] * np.random.randn()
    ax.plot(mp, color=COLORS[asset], linewidth=0.5, alpha=0.8, label='actual')
    ax.plot(sim, color='gray', linewidth=0.5, alpha=0.6, label='simulated ou')
    ax.set_title(f'{asset.lower()} -- actual vs simulated ou path')
    ax.set_xlabel('tick')
    ax.set_ylabel('price')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()


## section 9: variance ratio test

the variance ratio test (lo and mackinlay, 1988) directly tests whether price increments
are independent (random walk) by comparing variance at different horizons.

for a random walk, the variance of k-step returns equals k times the variance of 1-step returns:
    Var(k-step) = k * Var(1-step)

the variance ratio is:
    VR(k) = Var(k-step) / (k * Var(1-step))

VR = 1.0 -> consistent with random walk
VR > 1.0 -> positive autocorrelation (momentum / trending)
VR < 1.0 -> negative autocorrelation (mean reversion)

we compute VR at multiple horizons k = 2, 4, 8, 16, 32, 64 ticks.

the z-statistic for testing VR = 1 (under the null of independent increments) is:
    z(k) = sqrt(n) * (VR(k) - 1) / sqrt(2 * (2k-1) * (k-1) / (3k))

(this is the homoskedastic variance ratio statistic from lo and mackinlay 1988)


In [ ]:
def variance_ratio_test(series, horizons=None):
    """
    compute variance ratios and z-statistics for a range of horizons.
    
    VR(k) < 1 implies mean reversion at that horizon.
    VR(k) > 1 implies trending (momentum) at that horizon.
    z < -1.96 rejects the random walk null at 5% in favour of mean reversion.
    z > +1.96 rejects in favour of trending.
    """
    if horizons is None:
        horizons = [2, 4, 8, 16, 32, 64, 128]
    p = np.array(series, dtype=float)
    n = len(p) - 1
    increments = np.diff(p)
    var_1 = np.var(increments, ddof=1)
    results = []
    for k in horizons:
        k_step = p[k:] - p[:-k]
        var_k = np.var(k_step, ddof=1)
        vr = var_k / (k * var_1)
        # homoskedastic z-statistic (lo & mackinlay 1988 eq 2.5)
        phi = 2 * (2*k - 1) * (k - 1) / (3 * k)
        z = np.sqrt(n / k) * (vr - 1) / np.sqrt(phi)
        pval = 2 * (1 - stats.norm.cdf(abs(z)))
        results.append({'k': k, 'VR': vr, 'z': z, 'p': pval})
    return pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
vr_results = {}
for i, asset in enumerate(ASSETS):
    mp = asset_data[asset]['mid_price'].values
    vr_df = variance_ratio_test(mp)
    vr_results[asset] = vr_df

    ax = axes[i]
    bars = ax.bar(vr_df['k'].astype(str), vr_df['VR'], color=COLORS[asset], alpha=0.7, edgecolor='white')
    ax.axhline(1.0, color='black', linewidth=2, linestyle='--', label='VR=1 (random walk)')
    ax.set_title(f'{asset.lower()} -- variance ratio by horizon')
    ax.set_xlabel('horizon k (ticks)')
    ax.set_ylabel('variance ratio VR(k)')
    ax.legend()

    # colour bars that are statistically significant
    for bar, (_, row) in zip(bars, vr_df.iterrows()):
        if row['p'] < 0.05:
            bar.set_edgecolor('black')
            bar.set_linewidth(2.5)

plt.tight_layout()
plt.show()

print("variance ratio test results:")
print("(bars with thick black border are statistically significant rejections of the random walk)")
print()
for asset in ASSETS:
    print(f"\n{asset.lower()}")
    print(vr_results[asset].to_string(index=False, float_format=lambda x: f'{x:.4f}'))


## section 10: diffusion entropy

diffusion entropy measures the information content of the random component of price changes.

we estimate the differential entropy of the increment distribution:
    H_diff = -integral p(x) * log(p(x)) dx

for a normal distribution with standard deviation sigma:
    H_diff = 0.5 * log(2 * pi * e * sigma^2)  [nats]

a high entropy means the noise is spread out -- harder to predict.
a low entropy means increments are tightly clustered -- more predictable.

we also compute the excess entropy relative to the gaussian benchmark:
    excess_entropy = H_sample - H_gaussian(same sigma)

positive excess entropy means the distribution has heavier tails than gaussian
(more extreme moves, more unpredictability in the tails).
negative would mean the distribution is more concentrated (sub-gaussian).

we use a kernel density estimate (kde) to get the non-parametric entropy,
avoiding the gaussian assumption.


In [ ]:
def differential_entropy_kde(data, n_points=500):
    """
    estimate differential entropy using a gaussian kernel density estimate.
    
    we integrate -p(x) * log(p(x)) over the support of the distribution.
    this is the non-parametric version -- no distributional assumption.
    
    formula: H = -integral p(x) * ln(p(x)) dx
    units: nats (base-e logarithm)
    """
    data = np.array(data, dtype=float)
    kde = stats.gaussian_kde(data)
    x = np.linspace(data.min() - 3*data.std(), data.max() + 3*data.std(), n_points)
    p = kde(x)
    p = np.maximum(p, 1e-20)  # avoid log(0)
    h = -np.trapezoid(p * np.log(p), x)
    return h

def gaussian_entropy(sigma):
    """theoretical differential entropy of a gaussian with std sigma (nats)."""
    return 0.5 * np.log(2 * np.pi * np.e * sigma**2)

print("diffusion entropy analysis")
print("=" * 55)
entropy_results = {}
for asset in ASSETS:
    mp = asset_data[asset]['mid_price'].values
    increments = np.diff(mp)
    sigma = increments.std(ddof=1)

    h_kde = differential_entropy_kde(increments)
    h_gauss = gaussian_entropy(sigma)
    excess = h_kde - h_gauss

    entropy_results[asset] = {
        'h_kde': h_kde,
        'h_gauss': h_gauss,
        'excess': excess,
        'sigma': sigma,
    }

    print(f"\n{asset.lower()}")
    print(f"  non-parametric entropy  : {h_kde:.5f} nats")
    print(f"  gaussian benchmark      : {h_gauss:.5f} nats  (same sigma = {sigma:.4f})")
    print(f"  excess entropy          : {excess:+.5f} nats")
    if excess > 0.05:
        print(f"  interpretation          : heavier tails than gaussian -- more extreme moves than expected")
    elif excess < -0.05:
        print(f"  interpretation          : lighter tails than gaussian -- increments more predictable than noise suggests")
    else:
        print(f"  interpretation          : close to gaussian -- increment distribution well described by normal")


## section 11: drift-to-diffusion ratio and signal strength

the drift-to-diffusion ratio (ddr) quantifies how much of the price movement
is systematic (drift) versus random (noise).

    ddr = |mu| / sigma

this is mathematically equivalent to the absolute sharpe ratio per unit time.

over a horizon of T ticks, the expected price move from drift is |mu| * T,
while the expected random deviation grows as sigma * sqrt(T).
the "break-even" horizon is when signal equals noise:

    T_breakeven = (sigma / mu)^2  ticks

this tells us the minimum horizon over which drift is detectable above noise.

a related quantity is the information ratio: for trading, we care about the
expected return per unit of risk taken. this connects directly to the ddr.

note: if drift is not statistically significant (as per section 5), the ddr
should not be interpreted as a signal -- it may simply reflect sampling noise.


In [ ]:
print("drift-to-diffusion ratio analysis")
print("=" * 60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, asset in enumerate(ASSETS):
    mp = asset_data[asset]['mid_price'].values
    p = diffusion_params[asset]
    mu = abs(p['mu'])
    sigma = p['sigma']
    ddr = mu / sigma

    # signal-to-noise as function of horizon
    horizons = np.arange(1, 5001)
    signal = mu * horizons           # expected drift magnitude
    noise  = sigma * np.sqrt(horizons)
    snr    = signal / noise

    if mu > 0:
        t_be = (sigma / mu)**2
    else:
        t_be = np.inf

    ax = axes[i]
    ax.plot(horizons, snr, color=COLORS[asset], linewidth=2, label='signal / noise')
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.5, label='signal = noise')
    if np.isfinite(t_be) and t_be < 5000:
        ax.axvline(t_be, color='red', linestyle=':', linewidth=1.5, label=f'breakeven at {t_be:.0f} ticks')
    ax.set_title(f'{asset.lower()} -- signal-to-noise ratio vs horizon')
    ax.set_xlabel('horizon (ticks)')
    ax.set_ylabel('|mu|*T / (sigma*sqrt(T))')
    ax.set_ylim(0, max(snr.max()*1.1, 1.5))
    ax.legend()

    print(f"\n{asset.lower()}")
    print(f"  |drift| mu     : {mu:.8f} per tick")
    print(f"  diffusion sigma: {sigma:.6f} per tick")
    print(f"  ddr per tick   : {ddr:.8f}")
    print(f"  breakeven horizon : {t_be:.1f} ticks {'(within data range)' if t_be < 20000 else '(exceeds data range)'}")

plt.tight_layout()
plt.show()


## section 12: zero crossing frequency

a zero crossing occurs when the demeaned price series changes sign --
i.e., crosses from above the mean to below it, or vice versa.

the zero crossing rate (zcr) is the fraction of time steps where such a crossing occurs.

for an ou process with mean-reversion speed theta, the theoretical zero crossing rate is:
    zcr_theory = theta / pi  (for continuous time)

in discrete time with unit steps, faster mean reversion leads to more frequent crossings.

a high zcr means the process oscillates rapidly around its mean -- strong mean reversion signal.
a low zcr means the process wanders far from its mean before returning -- slow reversion.

we also look at the distribution of "excursion lengths" -- how long the process
stays on one side of the mean before crossing. this gives the expected holding period
for a mean-reversion trade.


note on the gap between observed and theoretical zcr:

the theoretical formula zcr = theta / pi comes from continuous-time ou theory. in discrete time
at our sampling frequency, the approximation breaks down. for emeralds with theta = 0.98
(nearly full reversion each tick), the continuous-time formula predicts zcr = 0.31, but the
observed zcr is only 0.032. the discrepancy arises because at the discrete level, the process
can revert "within" a tick without actually crossing zero. the observed zcr is the more
operationally meaningful number for strategy design: on average, the price stays on one side
of the equilibrium for 1/0.032 = 31 ticks before crossing.


In [ ]:
def zero_crossing_analysis(series, mean_override=None):
    """
    compute zero crossing statistics for a demeaned series.
    
    returns:
        zcr        : zero crossing rate (fraction of steps)
        excursions : list of lengths of runs above/below zero
        mean_excursion : expected excursion length (ticks)
    """
    p = np.array(series, dtype=float)
    mu = mean_override if mean_override is not None else p.mean()
    x = p - mu  # demeaned

    # crossings: sign changes
    signs = np.sign(x)
    signs[signs == 0] = 1  # treat exact-zero as positive
    crossings = np.diff(signs) != 0
    zcr = crossings.mean()

    # excursion lengths
    excursion_lengths = []
    current = 1
    for c in crossings:
        if c:
            excursion_lengths.append(current)
            current = 1
        else:
            current += 1
    excursion_lengths.append(current)

    return {
        'zcr': zcr,
        'excursions': np.array(excursion_lengths),
        'mean_excursion': np.mean(excursion_lengths),
        'median_excursion': np.median(excursion_lengths),
        'n_crossings': crossings.sum(),
        'mu': mu,
    }

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
zcr_results = {}
for i, asset in enumerate(ASSETS):
    mp = asset_data[asset]['mid_price'].values
    res = zero_crossing_analysis(mp, mean_override=ou_results[asset]['mu_eq'])
    zcr_results[asset] = res

    # demeaned series with crossings marked
    ax = axes[i, 0]
    x = mp - res['mu']
    ax.plot(x, color=COLORS[asset], linewidth=0.5, alpha=0.7, label='demeaned price')
    ax.axhline(0, color='black', linewidth=1.5, linestyle='--')
    ax.fill_between(np.arange(len(x)), x, 0, where=x > 0, alpha=0.2, color='green', label='above mean')
    ax.fill_between(np.arange(len(x)), x, 0, where=x < 0, alpha=0.2, color='red', label='below mean')
    ax.set_title(f'{asset.lower()} -- demeaned price (mean = ou equilibrium)')
    ax.set_xlabel('tick')
    ax.set_ylabel('price - equilibrium')
    ax.legend(fontsize=9)

    # excursion length distribution
    ax = axes[i, 1]
    exc = res['excursions']
    max_show = min(exc.max(), 200)
    ax.hist(exc[exc <= max_show], bins=40, color=COLORS[asset], alpha=0.7, edgecolor='white', density=True)
    ax.axvline(res['mean_excursion'], color='black', linestyle='--', linewidth=2,
               label=f'mean = {res["mean_excursion"]:.1f} ticks')
    ax.axvline(res['median_excursion'], color='navy', linestyle=':', linewidth=2,
               label=f'median = {res["median_excursion"]:.0f} ticks')
    ax.set_title(f'{asset.lower()} -- excursion length distribution (clipped at 200)')
    ax.set_xlabel('ticks above/below mean before crossing')
    ax.set_ylabel('density')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("zero crossing analysis")
print("=" * 55)
for asset in ASSETS:
    r = zcr_results[asset]
    theta = ou_results[asset]['theta']
    zcr_theory = theta / np.pi if theta > 0 else 0
    print(f"\n{asset.lower()}")
    print(f"  equilibrium (ou mu)      : {r['mu']:.4f}")
    print(f"  zero crossing rate (zcr) : {r['zcr']:.5f}  (fraction of steps)")
    print(f"  theoretical zcr (ou)     : {zcr_theory:.5f}  (theta/pi, continuous-time approximation)")
    print(f"  number of crossings      : {r['n_crossings']:,}")
    print(f"  mean excursion length    : {r['mean_excursion']:.2f} ticks")
    print(f"  median excursion length  : {r['median_excursion']:.0f} ticks")


## section 13: regime change detection via cusum

the cusum (cumulative sum) statistic detects structural breaks -- points where
the statistical properties of the series change suddenly.

we apply cusum to the demeaned price series. the statistic is:
    C(t) = max(0, C(t-1) + (x(t) - mu - k))   [upper cusum]
    C_lower(t) = max(0, C_lower(t-1) - (x(t) - mu + k))  [lower cusum]

where k is the "allowance" parameter (typically sigma/2).

when C(t) or C_lower(t) exceeds a threshold h (typically 5*sigma), we flag a regime change.

we also use rolling statistics (rolling mean and rolling std) as a simpler,
more interpretable alternative to detect slow drift in the equilibrium level.

limitations:
- cusum requires a reference mean and variance -- if these themselves shift, the detector lags
- in very short or very stable series, cusum may fire false alarms or miss slow changes
- we use heuristic thresholds; more rigorous approaches use average run length (arl) theory


In [ ]:
def cusum_detection(series, k_mult=0.5, h_mult=5, window=500):
    """
    apply the cusum algorithm to a series for changepoint detection.
    
    k = k_mult * sigma  (reference value / allowance)
    h = h_mult * sigma  (decision threshold)
    
    returns upper and lower cusum arrays, alarm indices, and rolling stats.
    """
    x = np.array(series, dtype=float)
    n = len(x)
    mu_ref = x[:window].mean()
    sigma_ref = x[:window].std(ddof=1)

    k = k_mult * sigma_ref
    h = h_mult * sigma_ref

    C_up   = np.zeros(n)
    C_down = np.zeros(n)

    for t in range(1, n):
        C_up[t]   = max(0, C_up[t-1]   + (x[t] - mu_ref - k))
        C_down[t] = max(0, C_down[t-1] - (x[t] - mu_ref + k))

    alarms_up   = np.where(C_up   > h)[0]
    alarms_down = np.where(C_down > h)[0]

    # rolling stats
    roll_mean = pd.Series(x).rolling(window).mean().values
    roll_std  = pd.Series(x).rolling(window).std().values

    return {
        'C_up': C_up, 'C_down': C_down,
        'alarms_up': alarms_up, 'alarms_down': alarms_down,
        'h': h, 'k': k, 'mu_ref': mu_ref, 'sigma_ref': sigma_ref,
        'roll_mean': roll_mean, 'roll_std': roll_std,
    }

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
cusum_results = {}
for i, asset in enumerate(ASSETS):
    mp = asset_data[asset]['mid_price'].values
    res = cusum_detection(mp)
    cusum_results[asset] = res

    # price with rolling mean
    ax = axes[i, 0]
    ax.plot(mp, color=COLORS[asset], linewidth=0.5, alpha=0.7, label='mid price')
    ax.plot(res['roll_mean'], color='black', linewidth=1.5, label='rolling mean (500 ticks)')
    ax.fill_between(np.arange(len(mp)),
                    res['roll_mean'] - 2*res['roll_std'],
                    res['roll_mean'] + 2*res['roll_std'],
                    alpha=0.15, color='gray', label='rolling mean +/- 2 std')
    ax.set_title(f'{asset.lower()} -- price with rolling statistics')
    ax.set_xlabel('tick')
    ax.set_ylabel('price')
    ax.legend(fontsize=9)

    # cusum chart
    ax = axes[i, 1]
    ax.plot(res['C_up'],   color='green', linewidth=1, label='cusum upper')
    ax.plot(res['C_down'], color='red',   linewidth=1, label='cusum lower')
    ax.axhline(res['h'], color='black', linestyle='--', linewidth=2, label=f'threshold h={res["h"]:.2f}')
    if len(res['alarms_up']) > 0:
        ax.axvline(res['alarms_up'][0], color='green', linestyle=':', linewidth=2,
                   label=f'first up alarm @ tick {res["alarms_up"][0]}')
    if len(res['alarms_down']) > 0:
        ax.axvline(res['alarms_down'][0], color='red', linestyle=':', linewidth=2,
                   label=f'first down alarm @ tick {res["alarms_down"][0]}')
    ax.set_title(f'{asset.lower()} -- cusum chart')
    ax.set_xlabel('tick')
    ax.set_ylabel('cusum statistic')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("cusum regime detection summary")
print("=" * 55)
for asset in ASSETS:
    r = cusum_results[asset]
    print(f"\n{asset.lower()}")
    print(f"  reference mean      : {r['mu_ref']:.4f}  (estimated from first 500 ticks)")
    print(f"  reference sigma     : {r['sigma_ref']:.4f}")
    print(f"  decision threshold h: {r['h']:.4f}  (= 5 * sigma)")
    print(f"  upper alarms        : {len(r['alarms_up'])} detected")
    print(f"  lower alarms        : {len(r['alarms_down'])} detected")
    if len(r['alarms_up']) > 0:
        print(f"  first upper alarm at tick {r['alarms_up'][0]}")
    if len(r['alarms_down']) > 0:
        print(f"  first lower alarm at tick {r['alarms_down'][0]}")


## section 14: spread and liquidity analysis

the bid-ask spread is the friction cost of trading. any signal must
overcome this friction to be profitable.

we examine:
- spread distribution per asset
- spread vs. price deviation (does spread widen when price is far from equilibrium?)
- trade price vs. mid-price deviation at time of trade (is there adverse selection?)

for emeralds the critical finding from the data is that the mean deviation from equilibrium
(0.13 price units) is 120x smaller than the mean bid-ask spread (15.74 price units). this means
the mid-price almost never strays far enough from 10000 to make a spread-crossing trade profitable.
the only viable strategy for emeralds is market-making (posting limit orders at the bid and ask
to collect the spread), not market-taking (hitting the bid/ask to trade on mean reversion).

a key metric is "spread-adjusted half-life" -- the minimum half-life at which a
mean-reversion strategy is profitable. if the expected reversion per half-life is
smaller than the spread, the strategy will lose money even if the signal is correct.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

print("spread and cost analysis")
print("=" * 60)
for i, asset in enumerate(ASSETS):
    df = asset_data[asset]
    mp = df['mid_price'].values
    spread = (df['ask_price_1'] - df['bid_price_1']).values
    ou_mu = ou_results[asset]['mu_eq']
    ou_hl = ou_results[asset]['half_life']
    ou_sigma = ou_results[asset]['sigma_ou']
    mean_spread = np.nanmean(spread)

    # expected reversion over half-life: starts at deviation d, goes to d/2
    # we enter when deviation = d, expect to exit when deviation = 0
    # expected gain = d (entry deviation)
    # cost = half spread to enter + half spread to exit = mean_spread
    # profit = d - mean_spread

    # minimum profitable deviation: d > mean_spread
    min_dev_for_profit = mean_spread
    # expected deviation magnitude based on ou
    expected_dev = ou_sigma * np.sqrt(2 / np.pi)  # mean abs of ou steady-state

    # spread-adjusted half-life break-even: what half-life maximises expected profit?
    # the key question: over half_life ticks, what is the expected reversion?
    # for ou: E[|p(t) - mu|] ~ |p(0) - mu| * exp(-theta * t) + sigma*sqrt(1/(2*theta))
    # the "edge" is the deviation minus the spread
    dev_at_entry = np.abs(mp - ou_mu)

    # scatter: |deviation| vs spread
    ax = axes[i, 0]
    ax.scatter(np.abs(mp - ou_mu), spread, color=COLORS[asset], s=2, alpha=0.1)
    ax.axvline(mean_spread, color='black', linestyle='--', linewidth=2,
               label=f'mean spread = {mean_spread:.2f}')
    ax.set_title(f'{asset.lower()} -- deviation vs spread')
    ax.set_xlabel('|price - ou equilibrium|')
    ax.set_ylabel('bid-ask spread')
    ax.legend(fontsize=9)

    # spread over time
    ax = axes[i, 1]
    ax.plot(spread, color=COLORS[asset], linewidth=0.5, alpha=0.7)
    ax.axhline(mean_spread, color='black', linestyle='--', linewidth=1.5, label=f'mean = {mean_spread:.2f}')
    ax.set_title(f'{asset.lower()} -- spread over time')
    ax.set_xlabel('tick')
    ax.set_ylabel('spread')
    ax.legend(fontsize=9)

    # trade price vs mid-price
    tr = asset_data[f'{asset}_trades']
    ax = axes[i, 2]
    if len(tr) > 0:
        # join trade to mid-price at closest tick
        tr_ticks = tr['tick'].values.astype(int)
        valid = (tr_ticks >= df.index.min()) & (tr_ticks <= df.index.max())
        tr_v = tr[valid]
        if len(tr_v) > 0:
            trade_dev = tr_v['price'].values - ou_mu
            ax.hist(trade_dev, bins=30, color=COLORS[asset], alpha=0.7, edgecolor='white')
            ax.axvline(0, color='black', linewidth=2, label=f'equilibrium = {ou_mu:.2f}')
            ax.set_title(f'{asset.lower()} -- trade price deviation from equilibrium')
            ax.set_xlabel('trade price - ou equilibrium')
            ax.set_ylabel('count')
            ax.legend(fontsize=9)

    # print stats
    print(f"\n{asset.lower()}")
    print(f"  mean bid-ask spread             : {mean_spread:.4f}")
    print(f"  ou equilibrium                  : {ou_mu:.4f}")
    print(f"  ou half-life                    : {ou_hl:.2f} ticks")
    print(f"  ou sigma (residual vol)         : {ou_sigma:.4f}")
    print(f"  min profitable deviation (>spread): {min_dev_for_profit:.4f}")
    print(f"  mean observed deviation |p-mu|  : {dev_at_entry.mean():.4f}")
    profitable_pct = (dev_at_entry > mean_spread).mean() * 100
    print(f"  fraction of ticks: deviation > spread: {profitable_pct:.1f}%")

plt.tight_layout()
plt.show()


## section 15: trading signals and entry/exit rules

this section synthesises all the above analyses into concrete, actionable signals.

we define:
- z-score: how many standard deviations the current price is from the ou equilibrium
  z = (p - mu_eq) / sigma_ou
- entry threshold: z > entry_z means the price is "too high" (short signal)
            z < -entry_z means the price is "too low" (long signal)
- exit threshold: z = 0 (return to equilibrium)

the choice of entry_z involves a tradeoff:
- higher entry_z: fewer signals but higher confidence and larger expected reversion
- lower entry_z: more signals but smaller expected profit per trade

we also show the cumulative pnl of a simple z-score strategy as a diagnostic.


In [ ]:
def z_score_signal(series, mu_eq, sigma_ou, entry_z=1.5, exit_z=0.3):
    """
    generate long/short signals based on z-score of demeaned price.
    
    signal logic:
        z < -entry_z  -> open long position (price is too low)
        z > +entry_z  -> open short position (price is too high)
        |z| < exit_z  -> close any open position (price returned to mean)
    
    returns a series of positions (+1 = long, -1 = short, 0 = flat).
    """
    p = np.array(series, dtype=float)
    z = (p - mu_eq) / sigma_ou
    positions = np.zeros(len(p))
    pos = 0
    for t in range(1, len(p)):
        if pos == 0:
            if z[t] < -entry_z:
                pos = 1   # enter long
            elif z[t] > entry_z:
                pos = -1  # enter short
        else:
            if abs(z[t]) < exit_z:
                pos = 0   # exit
        positions[t] = pos
    return positions, z

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
signal_summary = {}
for i, asset in enumerate(ASSETS):
    mp = asset_data[asset]['mid_price'].values
    spread = (asset_data[asset]['ask_price_1'] - asset_data[asset]['bid_price_1']).values
    mu_eq = ou_results[asset]['mu_eq']
    sigma_ou = ou_results[asset]['sigma_ou']
    mean_spread = np.nanmean(spread)

    # generate signals
    positions, z = z_score_signal(mp, mu_eq, sigma_ou, entry_z=1.5, exit_z=0.3)

    # compute pnl (ignoring spread cost first)
    raw_returns = np.diff(mp) * positions[:-1]

    # spread cost: paid at each entry/exit
    position_changes = np.diff(np.concatenate([[0], positions]))
    spread_cost = np.abs(position_changes) * mean_spread / 2  # half-spread per side
    net_returns = raw_returns - spread_cost[1:]

    cum_pnl_gross = np.cumsum(raw_returns)
    cum_pnl_net   = np.cumsum(net_returns)

    n_trades = (np.abs(position_changes) > 0).sum() // 2
    win_rate = (raw_returns[positions[:-1] != 0] > 0).mean() if (positions[:-1] != 0).any() else 0

    signal_summary[asset] = {
        'n_trades': n_trades,
        'total_gross_pnl': cum_pnl_gross[-1],
        'total_net_pnl': cum_pnl_net[-1],
        'win_rate': win_rate,
    }

    # z-score and signals
    ax = axes[i, 0]
    ax.plot(z, color=COLORS[asset], linewidth=0.5, alpha=0.8, label='z-score')
    ax.axhline(1.5,  color='red',   linestyle='--', linewidth=1.5, label='entry short (+1.5)')
    ax.axhline(-1.5, color='green', linestyle='--', linewidth=1.5, label='entry long (-1.5)')
    ax.axhline(0.3,  color='gray',  linestyle=':',  linewidth=1,   label='exit zone')
    ax.axhline(-0.3, color='gray',  linestyle=':',  linewidth=1)
    ax.set_title(f'{asset.lower()} -- z-score signal (entry |z|>1.5, exit |z|<0.3)')
    ax.set_xlabel('tick')
    ax.set_ylabel('z-score')
    ax.legend(fontsize=9)

    # cumulative pnl
    ax = axes[i, 1]
    ax.plot(cum_pnl_gross, color=COLORS[asset], linewidth=1.2, label='gross pnl')
    ax.plot(cum_pnl_net,   color='black',        linewidth=1.2, linestyle='--', label='net pnl (after spread cost)')
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.set_title(f'{asset.lower()} -- cumulative strategy pnl (z-score, entry 1.5)')
    ax.set_xlabel('tick')
    ax.set_ylabel('cumulative pnl (price units)')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("z-score strategy summary (entry at |z| > 1.5, exit at |z| < 0.3)")
print("=" * 65)
for asset in ASSETS:
    r = signal_summary[asset]
    print(f"\n{asset.lower()}")
    print(f"  number of completed round trips: {r['n_trades']}")
    print(f"  gross cumulative pnl           : {r['total_gross_pnl']:.2f}")
    print(f"  net cumulative pnl             : {r['total_net_pnl']:.2f}")
    print(f"  win rate (gross)               : {r['win_rate']*100:.1f}%")


## section 16: consolidated metrics table and summary

all key metrics in one place for direct comparison.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

print("consolidated diffusion metrics -- emeralds vs tomatoes")
print("=" * 75)

rows = []
for asset in ASSETS:
    mp = asset_data[asset]['mid_price'].values
    p = diffusion_params[asset]
    ou = ou_results[asset]
    hr = hurst_results[asset]
    er = entropy_results[asset]
    zcr = zcr_results[asset]
    adf = adf_results[asset]

    rows.append({
        'asset': asset.lower(),
        'mean price': f"{mp.mean():.2f}",
        'price std': f"{mp.std():.4f}",
        'drift mu (per tick)': f"{p['mu']:+.6f}",
        'diffusion sigma': f"{p['sigma']:.6f}",
        'ddr |mu|/sigma': f"{abs(p['mu'])/p['sigma']:.6f}",
        'hurst exponent H': f"{hr['H']:.4f}",
        'adf t-stat': f"{adf['t']:.4f}",
        'ou equilibrium': f"{ou['mu_eq']:.4f}",
        'ou theta': f"{ou['theta']:.6f}",
        'half-life (ticks)': f"{ou['half_life']:.2f}",
        'diffusion entropy (nats)': f"{er['h_kde']:.5f}",
        'excess entropy': f"{er['excess']:+.5f}",
        'zcr (fraction)': f"{zcr['zcr']:.5f}",
        'mean excursion (ticks)': f"{zcr['mean_excursion']:.2f}",
    })

df_summary = pd.DataFrame(rows).set_index('asset').T
print(df_summary.to_string())

print()
print("critical value reference:")
print("  adf: reject unit root at 5% if t < -2.86, at 1% if t < -3.43")
print("  hurst: H < 0.45 = mean reverting, H = 0.5 = random walk, H > 0.55 = trending")
print("  zcr: higher = faster mean reversion")
print("  ddr: smaller = noise dominates (harder to directionally trade)")


## section 17: limitations, confidence intervals, and what to be cautious about

this section is as important as the results. every estimate above has uncertainty,
and some conclusions are more reliable than others.

what we are confident about:

1. emeralds is an extremely stable mean-reverting asset. the evidence is overwhelming:
   - the adf test strongly rejects the unit root
   - the hurst exponent is well below 0.5
   - the ou equilibrium is sharply defined at 10000
   - zero crossing frequency is high
   - variance ratios are significantly below 1 at all horizons tested
   
2. the spread for both assets (~13-16 price units) is large relative to the price standard deviation
   of emeralds (~0.72). this means mean-reversion trades on emeralds are only profitable
   if entered at deviations substantially larger than the spread.

3. the drift component for both assets is not statistically distinguishable from zero.
   the drift-to-diffusion ratio is tiny, meaning directional (momentum) trading is not supported
   by the data at these time horizons.

a note on the hurst exponent results:

the r/s analysis returned H = 0.526 for emeralds (near random walk) and H = 1.017 for tomatoes
(which is impossible -- H is bounded between 0 and 1). these results should be discarded.
the failure is caused by two things: (1) both increment series have strong negative lag-1
autocorrelation (~-0.49 and -0.42), which violates the iid assumption of r/s analysis;
(2) the price series are heavily quantized (emeralds trades in discrete steps around 10000,
tomatoes in steps of 0.5), which distorts the range-to-std ratio at small window sizes.
all other diagnostics (adf, ou, variance ratio, zcr) are more reliable and internally consistent.

a note on the apparent contradiction between adf and ou results for tomatoes:

the adf test (t = -1.53) fails to reject a unit root for tomatoes, yet the ou regression
finds a statistically significant mean-reversion coefficient (p = 0.000003). this is not
a contradiction -- it reflects the low power of the adf test when mean reversion is slow
(theta = 0.002, half-life = 311 ticks). the adf test has well-known difficulty distinguishing
a near-unit-root process from a true unit root. the variance ratio test (vr < 1 at all horizons,
highly significant at short horizons) corroborates the ou finding: tomatoes is mean-reverting,
just slowly. the ou half-life confidence interval [219, 539 ticks] confirms the signal exists
but is wide, reflecting genuine estimation uncertainty.

what we are less certain about:

1. tomatoes shows some mean reversion, but with a longer half-life. our estimate depends
   heavily on the ou model being correctly specified. if the true process has
   time-varying volatility or a drifting equilibrium, the half-life estimate is unreliable.

2. the regime detection (cusum) is sensitive to the choice of reference period and thresholds.
   we used the first 500 ticks as our reference -- if the market had a "warmup" period there,
   our reference is biased.

3. the trading pnl shown in section 15 is not a reliable backtest. it does not account for:
   - position limits
   - market impact (our orders moving the price)
   - inability to trade at the exact mid-price (we always pay the spread)
   - the fact that liquidity may be unavailable at the prices we want

4. with only two days of data (20,000 ticks per asset), we have limited statistical power
   to distinguish between a slow random walk and genuine but sluggish mean reversion.
   the hurst exponent confidence interval is meaningful but not tight.

5. all models assume stationarity -- that the underlying data-generating process does not
   change over the observation window. any regime shift (new market participants, changed
   information flow) would invalidate all of the above estimates going forward.


note on the gap between observed and theoretical zcr:

the theoretical formula zcr = theta / pi comes from continuous-time ou theory. in discrete time
at our sampling frequency, the approximation breaks down. for emeralds with theta = 0.98
(nearly full reversion each tick), the continuous-time formula predicts zcr = 0.31, but the
observed zcr is only 0.032. the discrepancy arises because at the discrete level, the process
can revert "within" a tick without actually crossing zero. the observed zcr is the more
operationally meaningful number for strategy design: on average, the price stays on one side
of the equilibrium for 1/0.032 = 31 ticks before crossing.


In [ ]:
# confidence intervals for key parameters
print("confidence intervals (95%) for key parameters")
print("=" * 60)

for asset in ASSETS:
    ou = ou_results[asset]
    p = diffusion_params[asset]
    hr = hurst_results[asset]
    n = diffusion_params[asset]['n_increments']

    print(f"\n{asset.lower()}")
    
    # theta confidence interval from ou regression
    se_b = ou['se_b']
    b = ou['b']
    theta_lo = -(b + 1.96 * se_b)
    theta_hi = -(b - 1.96 * se_b)
    if theta_lo > 0 and theta_hi > 0:
        hl_lo = np.log(2) / theta_hi
        hl_hi = np.log(2) / theta_lo
    else:
        hl_lo, hl_hi = 0, np.inf
    print(f"  theta:     [{theta_lo:.6f}, {theta_hi:.6f}]")
    print(f"  half-life: [{hl_lo:.2f}, {hl_hi:.2f}] ticks  (95% ci)")

    # sigma confidence interval (chi-squared based)
    sigma = p['sigma']
    chi2_lo = stats.chi2.ppf(0.025, df=n-1)
    chi2_hi = stats.chi2.ppf(0.975, df=n-1)
    sigma_lo = sigma * np.sqrt((n-1) / chi2_hi)
    sigma_hi = sigma * np.sqrt((n-1) / chi2_lo)
    print(f"  sigma:     [{sigma_lo:.6f}, {sigma_hi:.6f}]  (95% ci)")

    # hurst confidence interval
    H = hr['H']
    se_H = hr['se']
    print(f"  hurst H:   [{H - 1.96*se_H:.4f}, {H + 1.96*se_H:.4f}]  (95% ci)")


## section 18: final interpretation and trading recommendations

emeralds:

emeralds is a textbook mean-reverting asset. the ou equilibrium is at exactly 10000,
the half-life is extremely short (fractions of a tick to a few ticks), the adf test
overwhelmingly rejects a unit root, and the hurst exponent confirms anti-persistence.

the practical challenge: the bid-ask spread (~15.7 units) is enormous relative to
the typical deviation from 10000. the mid-price rarely strays more than ~4 units from 10000.
a viable strategy requires catching the rare larger deviations (|z| > 2 or 3)
and executing quickly. latency and market access are the binding constraints here,
not signal quality.

recommendation: do not use a directional mean-reversion (market-taking) strategy for emeralds.
the spread completely absorbs any expected profit. the correct strategy is market-making:
post a bid slightly below 10000 and an ask slightly above 10000, and collect the spread as
inventory mean-reverts naturally. monitor for the rare excursions beyond 4 units (3 std) which
may signal genuine information and warrant position flattening rather than doubling down.

the z-score strategy confirms this: 630 trades generated 2520 gross pnl but -7395 net pnl
after spread costs. the gross win rate of 96.3% is impressive but irrelevant when every trade
loses more to the spread than it gains from mean reversion. the signal is real -- the cost structure
makes directional trading on it unprofitable. monitor for
deviations exceeding 2 standard deviations. the primary risk is adverse selection --
large deviations may be driven by genuine information, not noise.

tomatoes:

tomatoes is also mean-reverting but with much higher volatility (~19.7 units std),
a less stable equilibrium, and a longer half-life. the hurst exponent is lower than 0.5,
confirming anti-persistence, but the equilibrium level drifts over the two days.

the wider price range means larger deviations are common, and the spread (~13 units)
is smaller relative to typical deviation (~19 units). there is more "room" to profit
from mean reversion if the equilibrium is correctly estimated.

recommendation: use a rolling estimate of the ou equilibrium (rolling window of 500-1000 ticks)
rather than the full-sample mean. enter positions at z > 1.5 standard deviations from
the rolling mean. be cautious of regime changes -- the cusum chart may reveal when the
equilibrium shifts, at which point existing positions should be closed.
